In [ ]:
import customtkinter as ctk
from tkinter import filedialog, messagebox
from PyPDF2 import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import os


# ============================================================
# CUSTOMTKINTER SETTINGS
# ============================================================

ctk.set_appearance_mode("light")
ctk.set_default_color_theme("blue")


# ============================================================
# COLORS
# ============================================================

BG = "#F3F7FC"
WHITE = "#FFFFFF"

PRIMARY = "#2563EB"
PRIMARY_HOVER = "#1D4ED8"

LIGHT_BLUE = "#EAF2FF"

TEXT = "#172033"
SECONDARY_TEXT = "#64748B"

BORDER = "#DCE4EF"

SUCCESS = "#16A34A"
SUCCESS_BG = "#EAF8EF"

WARNING = "#F59E0B"

DANGER = "#DC2626"
DANGER_BG = "#FEF2F2"


# ============================================================
# GLOBAL VARIABLES
# ============================================================

resume_text = ""
resume_file = ""

resume_skills = []
job_skills = []

matched_skills = []
missing_skills = []

skill_score = 0
similarity_score = 0
final_score = 0


# ============================================================
# SKILLS DATABASE
# ============================================================

SKILLS = [
    "python",
    "java",
    "c++",
    "javascript",
    "html",
    "css",
    "sql",
    "mysql",
    "mongodb",

    "machine learning",
    "deep learning",
    "artificial intelligence",
    "data science",
    "data analysis",

    "natural language processing",
    "nlp",
    "computer vision",

    "tensorflow",
    "pytorch",
    "keras",
    "scikit-learn",

    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "opencv",

    "git",
    "github",
    "docker",

    "flask",
    "django",
    "fastapi",

    "react",
    "node.js",

    "aws",
    "azure",

    "power bi",
    "tableau",
    "excel"
]


# ============================================================
# MAIN WINDOW
# ============================================================

app = ctk.CTk()

app.title("AI Resume Screening System")

app.geometry("1350x850")

app.minsize(1100, 750)

app.configure(
    fg_color=BG
)


# ============================================================
# PDF TEXT EXTRACTION
# ============================================================

def extract_pdf_text(file_path):

    try:

        reader = PdfReader(file_path)

        text = ""

        for page in reader.pages:

            page_text = page.extract_text()

            if page_text:

                text += page_text + "\n"

        return text

    except Exception as error:

        messagebox.showerror(
            "PDF Error",
            f"Unable to read the PDF.\n\n{error}"
        )

        return ""


# ============================================================
# SKILL EXTRACTION
# ============================================================

def extract_skills(text):

    text = text.lower()

    found = []

    for skill in SKILLS:

        pattern = r"(?<!\w)" + re.escape(skill.lower()) + r"(?!\w)"

        if re.search(pattern, text):

            found.append(skill)

    return found


# ============================================================
# TF-IDF SIMILARITY
# ============================================================

def calculate_similarity(resume, job):

    try:

        documents = [
            resume,
            job
        ]

        vectorizer = TfidfVectorizer(
            stop_words="english"
        )

        matrix = vectorizer.fit_transform(
            documents
        )

        similarity = cosine_similarity(
            matrix[0:1],
            matrix[1:2]
        )

        return similarity[0][0] * 100

    except Exception:

        return 0


# ============================================================
# TAB NAVIGATION
# ============================================================

tab_names = [
    "Dashboard",
    "Resume",
    "Job Requirements",
    "AI Analysis",
    "Results"
]

current_tab_index = 0


def open_tab(tab_name):

    global current_tab_index

    current_tab_index = tab_names.index(
        tab_name
    )

    tabs.set(tab_name)

    update_navigation()


def next_tab():

    global current_tab_index

    if current_tab_index < len(tab_names) - 1:

        current_tab_index += 1

        tabs.set(
            tab_names[current_tab_index]
        )

        update_navigation()


def previous_tab():

    global current_tab_index

    if current_tab_index > 0:

        current_tab_index -= 1

        tabs.set(
            tab_names[current_tab_index]
        )

        update_navigation()


def update_navigation():

    if current_tab_index == 0:

        back_button.configure(
            state="disabled"
        )

    else:

        back_button.configure(
            state="normal"
        )

    if current_tab_index == len(tab_names) - 1:

        next_button.configure(
            state="disabled"
        )

    else:

        next_button.configure(
            state="normal"
        )


# ============================================================
# DASHBOARD UPDATE
# ============================================================

def update_dashboard():

    # Resume status

    if resume_text:

        dashboard_resume_value.configure(
            text="✓ Uploaded",
            text_color=SUCCESS
        )

        dashboard_resume_card.configure(
            fg_color=SUCCESS_BG
        )

    else:

        dashboard_resume_value.configure(
            text="Not Uploaded",
            text_color=DANGER
        )

        dashboard_resume_card.configure(
            fg_color=WHITE
        )

    # Job status

    job_description = job_textbox.get(
        "1.0",
        "end"
    ).strip()

    if job_description:

        dashboard_job_value.configure(
            text="✓ Added",
            text_color=SUCCESS
        )

        dashboard_job_card.configure(
            fg_color=SUCCESS_BG
        )

    else:

        dashboard_job_value.configure(
            text="Not Added",
            text_color=DANGER
        )

        dashboard_job_card.configure(
            fg_color=WHITE
        )


# ============================================================
# UPLOAD RESUME
# ============================================================

def upload_resume():

    global resume_text
    global resume_file

    file_path = filedialog.askopenfilename(
        title="Select Resume PDF",
        filetypes=[
            ("PDF Files", "*.pdf")
        ]
    )

    if not file_path:

        return

    extracted_text = extract_pdf_text(
        file_path
    )

    if not extracted_text.strip():

        messagebox.showwarning(
            "Resume Error",
            "No readable text was found in this PDF."
        )

        return

    resume_text = extracted_text

    resume_file = file_path

    filename = os.path.basename(
        file_path
    )

    resume_filename.configure(
        text=filename
    )

    resume_status.configure(
        text="✓ Resume uploaded successfully",
        text_color=SUCCESS
    )

    resume_textbox.delete(
        "1.0",
        "end"
    )

    resume_textbox.insert(
        "1.0",
        resume_text
    )

    analysis_resume_status.configure(
        text="✓ Resume ready",
        text_color=SUCCESS
    )

    update_dashboard()

    messagebox.showinfo(
        "Resume Uploaded",
        "The resume was successfully uploaded and processed."
    )


# ============================================================
# CLEAR RESUME
# ============================================================

def clear_resume():

    global resume_text
    global resume_file

    resume_text = ""

    resume_file = ""

    resume_filename.configure(
        text="No resume selected"
    )

    resume_status.configure(
        text="No resume uploaded",
        text_color=SECONDARY_TEXT
    )

    resume_textbox.delete(
        "1.0",
        "end"
    )

    analysis_resume_status.configure(
        text="✗ No resume",
        text_color=DANGER
    )

    update_dashboard()


# ============================================================
# ANALYZE RESUME
# ============================================================

def analyze_resume():

    global resume_skills
    global job_skills
    global matched_skills
    global missing_skills

    global skill_score
    global similarity_score
    global final_score

    # Check resume

    if not resume_text:

        messagebox.showwarning(
            "Resume Required",
            "Please upload a resume first."
        )

        open_tab("Resume")

        return

    # Get job description

    job_description = job_textbox.get(
        "1.0",
        "end"
    ).strip()

    if not job_description:

        messagebox.showwarning(
            "Job Requirements Required",
            "Please enter the job requirements first."
        )

        open_tab("Job Requirements")

        return

    # Update status

    analysis_status.configure(
        text="AI is analyzing the candidate...",
        text_color=PRIMARY
    )

    analyze_button.configure(
        state="disabled",
        text="ANALYZING..."
    )

    app.update()

    # --------------------------------------------------------
    # Extract skills
    # --------------------------------------------------------

    resume_skills = extract_skills(
        resume_text
    )

    job_skills = extract_skills(
        job_description
    )

    # --------------------------------------------------------
    # Match skills
    # --------------------------------------------------------

    matched_skills = [
        skill
        for skill in job_skills
        if skill in resume_skills
    ]

    missing_skills = [
        skill
        for skill in job_skills
        if skill not in resume_skills
    ]

    # --------------------------------------------------------
    # Calculate skill score
    # --------------------------------------------------------

    if job_skills:

        skill_score = (
            len(matched_skills)
            /
            len(job_skills)
        ) * 100

    else:

        skill_score = 0

    # --------------------------------------------------------
    # Calculate TF-IDF similarity
    # --------------------------------------------------------

    similarity_score = calculate_similarity(
        resume_text,
        job_description
    )

    # --------------------------------------------------------
    # Final score
    # --------------------------------------------------------

    final_score = (
        skill_score * 0.70
        +
        similarity_score * 0.30
    )

    # --------------------------------------------------------
    # Recommendation
    # --------------------------------------------------------

    if final_score >= 75:

        recommendation = "STRONG MATCH"

        recommendation_color = SUCCESS

    elif final_score >= 50:

        recommendation = "GOOD MATCH"

        recommendation_color = WARNING

    else:

        recommendation = "LOW MATCH"

        recommendation_color = DANGER

    # ========================================================
    # UPDATE ANALYSIS TAB
    # ========================================================

    analysis_status.configure(
        text="✓ Analysis completed successfully",
        text_color=SUCCESS
    )

    detected_resume_skills.configure(
        text=", ".join(
            skill.title()
            for skill in resume_skills
        )
        if resume_skills
        else "No skills detected"
    )

    detected_job_skills.configure(
        text=", ".join(
            skill.title()
            for skill in job_skills
        )
        if job_skills
        else "No skills detected"
    )

    # ========================================================
    # UPDATE RESULTS
    # ========================================================

    score_label.configure(
        text=f"{final_score:.1f}%"
    )

    skill_score_label.configure(
        text=f"{skill_score:.1f}%"
    )

    similarity_label.configure(
        text=f"{similarity_score:.1f}%"
    )

    recommendation_label.configure(
        text=recommendation,
        text_color=recommendation_color
    )

    score_progress.set(
        final_score / 100
    )

    # ========================================================
    # MATCHED SKILLS
    # ========================================================

    matched_textbox.delete(
        "1.0",
        "end"
    )

    if matched_skills:

        for skill in matched_skills:

            matched_textbox.insert(
                "end",
                f"✓  {skill.title()}\n"
            )

    else:

        matched_textbox.insert(
            "end",
            "No matching skills found."
        )

    # ========================================================
    # MISSING SKILLS
    # ========================================================

    missing_textbox.delete(
        "1.0",
        "end"
    )

    if missing_skills:

        for skill in missing_skills:

            missing_textbox.insert(
                "end",
                f"✗  {skill.title()}\n"
            )

    else:

        missing_textbox.insert(
            "end",
            "No major missing skills."
        )

    # ========================================================
    # FINISH
    # ========================================================

    analyze_button.configure(
        state="normal",
        text="RUN AI ANALYSIS"
    )

    dashboard_ai_value.configure(
        text="✓ Completed",
        text_color=SUCCESS
    )

    open_tab("Results")


# ============================================================
# CLEAR ALL
# ============================================================

def clear_all():

    global resume_text
    global resume_file

    global resume_skills
    global job_skills

    global matched_skills
    global missing_skills

    global skill_score
    global similarity_score
    global final_score

    resume_text = ""

    resume_file = ""

    resume_skills = []

    job_skills = []

    matched_skills = []

    missing_skills = []

    skill_score = 0

    similarity_score = 0

    final_score = 0

    # Resume

    resume_filename.configure(
        text="No resume selected"
    )

    resume_status.configure(
        text="No resume uploaded",
        text_color=SECONDARY_TEXT
    )

    resume_textbox.delete(
        "1.0",
        "end"
    )

    # Job

    job_textbox.delete(
        "1.0",
        "end"
    )

    # Analysis

    analysis_resume_status.configure(
        text="✗ No resume",
        text_color=DANGER
    )

    analysis_status.configure(
        text="Waiting for analysis",
        text_color=SECONDARY_TEXT
    )

    detected_resume_skills.configure(
        text="No analysis yet"
    )

    detected_job_skills.configure(
        text="No analysis yet"
    )

    # Results

    score_label.configure(
        text="0%"
    )

    skill_score_label.configure(
        text="0%"
    )

    similarity_label.configure(
        text="0%"
    )

    recommendation_label.configure(
        text="WAITING",
        text_color=SECONDARY_TEXT
    )

    score_progress.set(0)

    matched_textbox.delete(
        "1.0",
        "end"
    )

    missing_textbox.delete(
        "1.0",
        "end"
    )

    dashboard_ai_value.configure(
        text="Ready",
        text_color=PRIMARY
    )

    update_dashboard()

    open_tab("Dashboard")


# ============================================================
# HEADER
# ============================================================

header = ctk.CTkFrame(
    app,
    fg_color=WHITE,
    corner_radius=0,
    height=85
)

header.pack(
    fill="x"
)

header.pack_propagate(False)


header_left = ctk.CTkFrame(
    header,
    fg_color="transparent"
)

header_left.pack(
    side="left",
    padx=35
)


header_title = ctk.CTkLabel(
    header_left,
    text="AI Resume Screening",
    font=ctk.CTkFont(
        size=25,
        weight="bold"
    ),
    text_color=TEXT
)

header_title.pack(
    anchor="w"
)


header_subtitle = ctk.CTkLabel(
    header_left,
    text="Intelligent Candidate & Job Matching System",
    font=ctk.CTkFont(
        size=13
    ),
    text_color=SECONDARY_TEXT
)

header_subtitle.pack(
    anchor="w"
)


header_status = ctk.CTkLabel(
    header,
    text="●  AI ENGINE READY",
    font=ctk.CTkFont(
        size=13,
        weight="bold"
    ),
    text_color=SUCCESS
)

header_status.pack(
    side="right",
    padx=35
)


# ============================================================
# TAB VIEW
# ============================================================

tabs = ctk.CTkTabview(
    app,

    fg_color=BG,

    corner_radius=15,

    segmented_button_fg_color=WHITE,

    segmented_button_selected_color=PRIMARY,

    segmented_button_selected_hover_color=PRIMARY_HOVER,

    segmented_button_unselected_color=WHITE,

    segmented_button_unselected_hover_color=LIGHT_BLUE,

    text_color=TEXT
)

tabs.pack(
    fill="both",
    expand=True,
    padx=25,
    pady=(20, 10)
)


# ============================================================
# CREATE TABS
# ============================================================

tabs.add("Dashboard")

tabs.add("Resume")

tabs.add("Job Requirements")

tabs.add("AI Analysis")

tabs.add("Results")


# ============================================================
# MAKE TABS WIDER
# Compatible with your CustomTkinter version
# ============================================================

tabs._segmented_button.configure(
    width=210,
    height=55,
    font=ctk.CTkFont(
        size=13,
        weight="bold"
    )
)


# ============================================================
# DASHBOARD TAB
# ============================================================

dashboard_tab = tabs.tab(
    "Dashboard"
)


dashboard_heading = ctk.CTkLabel(
    dashboard_tab,
    text="Welcome Back 👋",
    font=ctk.CTkFont(
        size=30,
        weight="bold"
    ),
    text_color=TEXT
)

dashboard_heading.pack(
    anchor="w",
    padx=35,
    pady=(30, 3)
)


dashboard_subtitle = ctk.CTkLabel(
    dashboard_tab,
    text="Start a new candidate screening or continue your current analysis.",
    font=ctk.CTkFont(
        size=14
    ),
    text_color=SECONDARY_TEXT
)

dashboard_subtitle.pack(
    anchor="w",
    padx=35
)


# ============================================================
# DASHBOARD CARDS
# ============================================================

dashboard_cards = ctk.CTkFrame(
    dashboard_tab,
    fg_color="transparent"
)

dashboard_cards.pack(
    fill="x",
    padx=30,
    pady=30
)


def create_dashboard_card(
    parent,
    title,
    initial,
    command
):

    card = ctk.CTkFrame(
        parent,
        fg_color=WHITE,
        corner_radius=16,
        border_width=1,
        border_color=BORDER
    )

    card.pack(
        side="left",
        fill="both",
        expand=True,
        padx=7
    )

    title_label = ctk.CTkLabel(
        card,
        text=title,
        font=ctk.CTkFont(
            size=13,
            weight="bold"
        ),
        text_color=SECONDARY_TEXT
    )

    title_label.pack(
        anchor="w",
        padx=22,
        pady=(20, 5)
    )

    value_label = ctk.CTkLabel(
        card,
        text=initial,
        font=ctk.CTkFont(
            size=21,
            weight="bold"
        ),
        text_color=TEXT
    )

    value_label.pack(
        anchor="w",
        padx=22,
        pady=(0, 20)
    )

    button = ctk.CTkButton(
        card,
        text="Open →",
        height=30,
        width=90,
        fg_color=LIGHT_BLUE,
        hover_color="#D8E7FF",
        text_color=PRIMARY,
        command=command
    )

    button.pack(
        anchor="w",
        padx=22,
        pady=(0, 18)
    )

    return card, value_label


dashboard_resume_card, dashboard_resume_value = create_dashboard_card(
    dashboard_cards,
    "RESUME",
    "Not Uploaded",
    lambda: open_tab("Resume")
)


dashboard_job_card, dashboard_job_value = create_dashboard_card(
    dashboard_cards,
    "JOB REQUIREMENTS",
    "Not Added",
    lambda: open_tab("Job Requirements")
)


dashboard_ai_card, dashboard_ai_value = create_dashboard_card(
    dashboard_cards,
    "AI ANALYSIS",
    "Ready",
    lambda: open_tab("AI Analysis")
)


# ============================================================
# START SCREENING CARD
# ============================================================

start_card = ctk.CTkFrame(
    dashboard_tab,
    fg_color=LIGHT_BLUE,
    corner_radius=18
)

start_card.pack(
    fill="x",
    padx=35,
    pady=(0, 20)
)


start_title = ctk.CTkLabel(
    start_card,
    text="Ready to screen a candidate?",
    font=ctk.CTkFont(
        size=20,
        weight="bold"
    ),
    text_color=TEXT
)

start_title.pack(
    side="left",
    padx=25,
    pady=25
)


start_button = ctk.CTkButton(
    start_card,
    text="START NEW SCREENING  →",
    width=230,
    height=45,
    font=ctk.CTkFont(
        size=13,
        weight="bold"
    ),
    fg_color=PRIMARY,
    hover_color=PRIMARY_HOVER,
    command=lambda: open_tab("Resume")
)

start_button.pack(
    side="right",
    padx=25
)


# ============================================================
# WORKFLOW CARD
# ============================================================

instruction_card = ctk.CTkFrame(
    dashboard_tab,
    fg_color=WHITE,
    corner_radius=16,
    border_width=1,
    border_color=BORDER
)

instruction_card.pack(
    fill="both",
    expand=True,
    padx=35,
    pady=(0, 20)
)


instruction_title = ctk.CTkLabel(
    instruction_card,
    text="Screening Workflow",
    font=ctk.CTkFont(
        size=19,
        weight="bold"
    ),
    text_color=TEXT
)

instruction_title.pack(
    anchor="w",
    padx=25,
    pady=(20, 10)
)


workflow = (
    "01   Upload the candidate's resume\n\n"
    "02   Enter the job requirements\n\n"
    "03   Run the AI analysis\n\n"
    "04   Review the candidate screening results"
)


workflow_label = ctk.CTkLabel(
    instruction_card,
    text=workflow,
    justify="left",
    font=ctk.CTkFont(
        size=14
    ),
    text_color=SECONDARY_TEXT
)

workflow_label.pack(
    anchor="w",
    padx=25
)


# ============================================================
# RESUME TAB
# ============================================================

resume_tab = tabs.tab(
    "Resume"
)


resume_heading = ctk.CTkLabel(
    resume_tab,
    text="Candidate Resume",
    font=ctk.CTkFont(
        size=28,
        weight="bold"
    ),
    text_color=TEXT
)

resume_heading.pack(
    anchor="w",
    padx=35,
    pady=(25, 5)
)


resume_subtitle = ctk.CTkLabel(
    resume_tab,
    text="Upload and process the candidate's PDF resume.",
    font=ctk.CTkFont(
        size=14
    ),
    text_color=SECONDARY_TEXT
)

resume_subtitle.pack(
    anchor="w",
    padx=35
)


# ============================================================
# UPLOAD CARD
# ============================================================

resume_upload_card = ctk.CTkFrame(
    resume_tab,
    fg_color=WHITE,
    corner_radius=16,
    border_width=1,
    border_color=BORDER
)

resume_upload_card.pack(
    fill="x",
    padx=35,
    pady=25
)


upload_button = ctk.CTkButton(
    resume_upload_card,
    text="＋  UPLOAD PDF",
    width=190,
    height=48,
    font=ctk.CTkFont(
        size=13,
        weight="bold"
    ),
    fg_color=PRIMARY,
    hover_color=PRIMARY_HOVER,
    command=upload_resume
)

upload_button.pack(
    side="left",
    padx=25,
    pady=22
)


resume_filename = ctk.CTkLabel(
    resume_upload_card,
    text="No resume selected",
    font=ctk.CTkFont(
        size=14,
        weight="bold"
    ),
    text_color=TEXT
)

resume_filename.pack(
    side="left",
    padx=10
)


resume_status = ctk.CTkLabel(
    resume_upload_card,
    text="No resume uploaded",
    text_color=SECONDARY_TEXT
)

resume_status.pack(
    side="left",
    padx=15
)


clear_resume_button = ctk.CTkButton(
    resume_upload_card,
    text="Clear",
    width=90,
    fg_color="#E8EDF5",
    hover_color="#D8E0EC",
    text_color=TEXT,
    command=clear_resume
)

clear_resume_button.pack(
    side="right",
    padx=25
)


# ============================================================
# RESUME PREVIEW
# ============================================================

resume_text_card = ctk.CTkFrame(
    resume_tab,
    fg_color=WHITE,
    corner_radius=16,
    border_width=1,
    border_color=BORDER
)

resume_text_card.pack(
    fill="both",
    expand=True,
    padx=35,
    pady=(0, 20)
)


resume_text_title = ctk.CTkLabel(
    resume_text_card,
    text="Resume Processing Preview",
    font=ctk.CTkFont(
        size=18,
        weight="bold"
    ),
    text_color=TEXT
)

resume_text_title.pack(
    anchor="w",
    padx=25,
    pady=(20, 10)
)


resume_textbox = ctk.CTkTextbox(
    resume_text_card,
    fg_color="#F8FAFC",
    text_color=TEXT,
    font=ctk.CTkFont(
        size=13
    ),
    corner_radius=10
)

resume_textbox.pack(
    fill="both",
    expand=True,
    padx=25,
    pady=(0, 20)
)


# ============================================================
# JOB REQUIREMENTS TAB
# ============================================================

job_tab = tabs.tab(
    "Job Requirements"
)


job_heading = ctk.CTkLabel(
    job_tab,
    text="Job Requirements",
    font=ctk.CTkFont(
        size=28,
        weight="bold"
    ),
    text_color=TEXT
)

job_heading.pack(
    anchor="w",
    padx=35,
    pady=(25, 5)
)


job_subtitle = ctk.CTkLabel(
    job_tab,
    text="Describe the position and list the skills required for the candidate.",
    font=ctk.CTkFont(
        size=14
    ),
    text_color=SECONDARY_TEXT
)

job_subtitle.pack(
    anchor="w",
    padx=35
)


job_card = ctk.CTkFrame(
    job_tab,
    fg_color=WHITE,
    corner_radius=16,
    border_width=1,
    border_color=BORDER
)

job_card.pack(
    fill="both",
    expand=True,
    padx=35,
    pady=25
)


job_textbox = ctk.CTkTextbox(
    job_card,
    fg_color="#F8FAFC",
    text_color=TEXT,
    font=ctk.CTkFont(
        size=14
    ),
    corner_radius=10
)

job_textbox.pack(
    fill="both",
    expand=True,
    padx=25,
    pady=25
)


job_textbox.insert(
    "1.0",
    "Example:\n\n"
    "We are looking for a Python Developer with experience "
    "in machine learning and data analysis.\n\n"
    "Required Skills:\n"
    "Python, SQL, Pandas, NumPy, Scikit-learn, "
    "Machine Learning, GitHub and NLP."
)


job_bottom = ctk.CTkFrame(
    job_card,
    fg_color="transparent"
)

job_bottom.pack(
    fill="x",
    padx=25,
    pady=(0, 20)
)


clear_job_button = ctk.CTkButton(
    job_bottom,
    text="Clear",
    width=100,
    fg_color="#E8EDF5",
    hover_color="#D8E0EC",
    text_color=TEXT,
    command=lambda: job_textbox.delete(
        "1.0",
        "end"
    )
)

clear_job_button.pack(
    side="right"
)


# ============================================================
# AI ANALYSIS TAB
# ============================================================

analysis_tab = tabs.tab(
    "AI Analysis"
)


analysis_heading = ctk.CTkLabel(
    analysis_tab,
    text="AI Resume Analysis",
    font=ctk.CTkFont(
        size=28,
        weight="bold"
    ),
    text_color=TEXT
)

analysis_heading.pack(
    anchor="w",
    padx=35,
    pady=(25, 5)
)


analysis_subtitle = ctk.CTkLabel(
    analysis_tab,
    text="Compare the candidate profile against the selected job requirements.",
    font=ctk.CTkFont(
        size=14
    ),
    text_color=SECONDARY_TEXT
)

analysis_subtitle.pack(
    anchor="w",
    padx=35
)


# ============================================================
# ANALYSIS STATUS
# ============================================================

analysis_status_card = ctk.CTkFrame(
    analysis_tab,
    fg_color=WHITE,
    corner_radius=16,
    border_width=1,
    border_color=BORDER
)

analysis_status_card.pack(
    fill="x",
    padx=35,
    pady=25
)


analysis_resume_status = ctk.CTkLabel(
    analysis_status_card,
    text="✗ No resume",
    font=ctk.CTkFont(
        size=14,
        weight="bold"
    ),
    text_color=DANGER
)

analysis_resume_status.pack(
    side="left",
    padx=25,
    pady=20
)


analysis_status = ctk.CTkLabel(
    analysis_status_card,
    text="Waiting for analysis",
    font=ctk.CTkFont(
        size=14,
        weight="bold"
    ),
    text_color=SECONDARY_TEXT
)

analysis_status.pack(
    side="right",
    padx=25
)


# ============================================================
# ANALYZE BUTTON
# ============================================================

analyze_button = ctk.CTkButton(
    analysis_tab,
    text="RUN AI ANALYSIS",
    width=300,
    height=55,
    font=ctk.CTkFont(
        size=15,
        weight="bold"
    ),
    fg_color=PRIMARY,
    hover_color=PRIMARY_HOVER,
    command=analyze_resume
)

analyze_button.pack(
    pady=5
)


# ============================================================
# DETECTED SKILLS
# ============================================================

skills_card = ctk.CTkFrame(
    analysis_tab,
    fg_color=WHITE,
    corner_radius=16,
    border_width=1,
    border_color=BORDER
)

skills_card.pack(
    fill="both",
    expand=True,
    padx=35,
    pady=25
)


skills_title = ctk.CTkLabel(
    skills_card,
    text="Detected Skills",
    font=ctk.CTkFont(
        size=19,
        weight="bold"
    ),
    text_color=TEXT
)

skills_title.pack(
    anchor="w",
    padx=25,
    pady=(20, 15)
)


candidate_skill_title = ctk.CTkLabel(
    skills_card,
    text="Candidate Skills",
    font=ctk.CTkFont(
        size=13,
        weight="bold"
    ),
    text_color=SECONDARY_TEXT
)

candidate_skill_title.pack(
    anchor="w",
    padx=25
)


detected_resume_skills = ctk.CTkLabel(
    skills_card,
    text="No analysis yet",
    wraplength=1100,
    justify="left",
    anchor="w",
    text_color=TEXT
)

detected_resume_skills.pack(
    anchor="w",
    padx=25,
    pady=(5, 20)
)


job_skill_title = ctk.CTkLabel(
    skills_card,
    text="Required Job Skills",
    font=ctk.CTkFont(
        size=13,
        weight="bold"
    ),
    text_color=SECONDARY_TEXT
)

job_skill_title.pack(
    anchor="w",
    padx=25
)


detected_job_skills = ctk.CTkLabel(
    skills_card,
    text="No analysis yet",
    wraplength=1100,
    justify="left",
    anchor="w",
    text_color=TEXT
)

detected_job_skills.pack(
    anchor="w",
    padx=25,
    pady=(5, 20)
)


# ============================================================
# RESULTS TAB
# ============================================================

results_tab = tabs.tab(
    "Results"
)


results_heading = ctk.CTkLabel(
    results_tab,
    text="Screening Results",
    font=ctk.CTkFont(
        size=28,
        weight="bold"
    ),
    text_color=TEXT
)

results_heading.pack(
    pady=(20, 3)
)


results_subtitle = ctk.CTkLabel(
    results_tab,
    text="AI-generated candidate compatibility report",
    font=ctk.CTkFont(
        size=14
    ),
    text_color=SECONDARY_TEXT
)

results_subtitle.pack()


# ============================================================
# SCORE CARD
# ============================================================

score_card = ctk.CTkFrame(
    results_tab,
    fg_color=WHITE,
    corner_radius=18,
    border_width=1,
    border_color=BORDER
)

score_card.pack(
    fill="x",
    padx=35,
    pady=20
)


score_label = ctk.CTkLabel(
    score_card,
    text="0%",
    font=ctk.CTkFont(
        size=48,
        weight="bold"
    ),
    text_color=PRIMARY
)

score_label.pack(
    pady=(15, 0)
)


score_title = ctk.CTkLabel(
    score_card,
    text="OVERALL MATCH SCORE",
    font=ctk.CTkFont(
        size=12,
        weight="bold"
    ),
    text_color=SECONDARY_TEXT
)

score_title.pack(
    pady=(0, 10)
)


score_progress = ctk.CTkProgressBar(
    score_card,
    width=650,
    height=12,
    fg_color="#E3EAF3",
    progress_color=PRIMARY
)

score_progress.set(0)

score_progress.pack(
    pady=(0, 20)
)


# ============================================================
# STATISTICS
# ============================================================

stats_frame = ctk.CTkFrame(
    results_tab,
    fg_color="transparent"
)

stats_frame.pack(
    fill="x",
    padx=35
)


def create_stat_card(
    parent,
    title,
    value
):

    card = ctk.CTkFrame(
        parent,
        fg_color=WHITE,
        corner_radius=15,
        border_width=1,
        border_color=BORDER
    )

    card.pack(
        side="left",
        fill="both",
        expand=True,
        padx=7
    )

    title_label = ctk.CTkLabel(
        card,
        text=title,
        font=ctk.CTkFont(
            size=12,
            weight="bold"
        ),
        text_color=SECONDARY_TEXT
    )

    title_label.pack(
        pady=(15, 3)
    )

    value_label = ctk.CTkLabel(
        card,
        text=value,
        font=ctk.CTkFont(
            size=23,
            weight="bold"
        ),
        text_color=TEXT
    )

    value_label.pack(
        pady=(0, 15)
    )

    return value_label


skill_score_label = create_stat_card(
    stats_frame,
    "SKILL MATCH",
    "0%"
)


similarity_label = create_stat_card(
    stats_frame,
    "AI SIMILARITY",
    "0%"
)


# ============================================================
# RECOMMENDATION
# ============================================================

recommendation_label = ctk.CTkLabel(
    results_tab,
    text="WAITING",
    font=ctk.CTkFont(
        size=21,
        weight="bold"
    ),
    text_color=SECONDARY_TEXT
)

recommendation_label.pack(
    pady=15
)


# ============================================================
# MATCHED / MISSING SKILLS
# ============================================================

details_frame = ctk.CTkFrame(
    results_tab,
    fg_color="transparent"
)

details_frame.pack(
    fill="both",
    expand=True,
    padx=35,
    pady=(0, 15)
)


# ------------------------------------------------------------
# MATCHED
# ------------------------------------------------------------

matched_card = ctk.CTkFrame(
    details_frame,
    fg_color=WHITE,
    corner_radius=15,
    border_width=1,
    border_color=BORDER
)

matched_card.pack(
    side="left",
    fill="both",
    expand=True,
    padx=(0, 8)
)


matched_title = ctk.CTkLabel(
    matched_card,
    text="✓  Matched Skills",
    font=ctk.CTkFont(
        size=16,
        weight="bold"
    ),
    text_color=SUCCESS
)

matched_title.pack(
    anchor="w",
    padx=20,
    pady=(15, 10)
)


matched_textbox = ctk.CTkTextbox(
    matched_card,
    fg_color="#F8FAFC",
    text_color=TEXT,
    font=ctk.CTkFont(
        size=13
    )
)

matched_textbox.pack(
    fill="both",
    expand=True,
    padx=20,
    pady=(0, 15)
)


# ------------------------------------------------------------
# MISSING
# ------------------------------------------------------------

missing_card = ctk.CTkFrame(
    details_frame,
    fg_color=WHITE,
    corner_radius=15,
    border_width=1,
    border_color=BORDER
)

missing_card.pack(
    side="right",
    fill="both",
    expand=True,
    padx=(8, 0)
)


missing_title = ctk.CTkLabel(
    missing_card,
    text="✗  Missing Skills",
    font=ctk.CTkFont(
        size=16,
        weight="bold"
    ),
    text_color=DANGER
)

missing_title.pack(
    anchor="w",
    padx=20,
    pady=(15, 10)
)


missing_textbox = ctk.CTkTextbox(
    missing_card,
    fg_color="#F8FAFC",
    text_color=TEXT,
    font=ctk.CTkFont(
        size=13
    )
)

missing_textbox.pack(
    fill="both",
    expand=True,
    padx=20,
    pady=(0, 15)
)


# ============================================================
# BOTTOM NAVIGATION
# ============================================================

navigation = ctk.CTkFrame(
    app,
    fg_color=WHITE,
    height=70,
    corner_radius=0
)

navigation.pack(
    fill="x"
)

navigation.pack_propagate(False)


back_button = ctk.CTkButton(
    navigation,
    text="←  Back",
    width=150,
    height=42,
    fg_color="#E8EDF5",
    hover_color="#D8E0EC",
    text_color=TEXT,
    command=previous_tab
)

back_button.pack(
    side="left",
    padx=30,
    pady=14
)


next_button = ctk.CTkButton(
    navigation,
    text="Next  →",
    width=150,
    height=42,
    fg_color=PRIMARY,
    hover_color=PRIMARY_HOVER,
    command=next_tab
)

next_button.pack(
    side="right",
    padx=30,
    pady=14
)


new_screening_button = ctk.CTkButton(
    navigation,
    text="＋  New Screening",
    width=180,
    height=40,
    fg_color=LIGHT_BLUE,
    hover_color="#D9E8FF",
    text_color=PRIMARY,
    command=clear_all
)

new_screening_button.pack(
    pady=15
)


# ============================================================
# INITIAL STATE
# ============================================================

update_dashboard()

update_navigation()

tabs.set("Dashboard")


# ============================================================
# START APPLICATION
# ============================================================

app.mainloop()